# MSDS 565 — Unit 1, Notebook 2: Preprocessing
**Team DataStars**

Builds a clean, model-ready dataset from the raw CSV and exports the 80/20 split for
Notebook 3. Every decision below carries a markdown explanation, grounded in the
findings from Notebook 1's EDA — the duplicate-row count, the mileage sentinel value,
the `"--"` placeholder, and everything else documented in that notebook's Data Quality
Register. Raw data is read fresh at the top; nothing carries over from the EDA notebook.


In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Same convention as Notebook 1 — each team member points this at their own local copy.
DATA_PATH = Path(r"C:\Users\cto\Desktop\MSDS-565\First-Project\CarSales_Large.csv")
OUTPUT_DIR = DATA_PATH.parent  # local class data folder — not committed to the repo

df = pd.read_csv(DATA_PATH, low_memory=False)
n_raw = len(df)
print(f"Raw shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


## 1. Dropping

### 1.1 Columns
Carried over directly from the EDA's Section 3 ("Which columns are useless") and the
Section 7 register — every column below is either a pure identifier, free text with no
tabular signal, entirely empty, a placeholder-only column, a verbatim restatement of
another column, or (in `savings_amount`'s case) computed directly from `price` and
therefore label leakage. `trimId` is added here too: at ~23,800 distinct values it's
effectively a per-listing identifier, and its information is already carried by
`trim_name`/`model_name`.


In [ ]:
drop_cols = [
    "Unnamed: 0",             # row-index artifact from a previous export
    "listing_id", "sp_id",    # unique identifiers -- cannot generalize
    "trimId",                 # near-unique per listing; redundant with trim_name/model_name
    "main_picture_url",       # image URL -- no tabular signal without image modeling
    "description",            # free-text ad copy -- needs NLP, out of scope here
    "combine_fuel_economy", "is_certified", "vehicle_damage_category",  # 100% missing
    "bed_height",              # only non-null value is the placeholder "--"
    "wheel_system_display",    # verbatim restatement of wheel_system
    "dealer_zip",              # redundant with latitude/longitude for location
    "savings_amount",          # derived from price -> label leakage
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)
print(f"Dropped {len(drop_cols)} columns: {drop_cols}")
print(f"Shape after column drops: {df.shape}")


**What just happened:** the raw file starts at 66 columns; dropping these 13 leaves 53. Nothing about the *data* changed here -- this only removes whole columns, so the row count is still the full 302,590.

### 1.2 Rows -- exact duplicates
1,210 rows are exact duplicates of another row once the ID columns are ignored
(confirmed in the EDA's Section 4.5). Keeping both copies risks the same listing
appearing in both the train and test split, artificially inflating test performance --
they're dropped here, before the split, rather than salvaged.


In [ ]:
n_before_dupes = len(df)
df = df.drop_duplicates()
n_dupes_dropped = n_before_dupes - len(df)
print(f"Dropped {n_dupes_dropped:,} exact-duplicate rows")


**Running total:** 302,590 raw rows -> 301,380 after removing the 1,210 duplicates. Column count is unchanged; only the row count moves here.

## 2. Data types

### 2.1 The `"--"` placeholder
`"--"` is used across roughly a dozen columns to mean "not reported," but pandas'
`isna()` doesn't recognize it as missing (EDA Section 4.1). Replacing it with a real
`NaN` here means every downstream missing-value check and imputation step in this
notebook sees the true picture, instead of quietly undercounting missingness.


In [ ]:
obj_cols_initial = df.select_dtypes("object").columns
placeholder_hits = 0
for c in obj_cols_initial:
    mask = df[c] == "--"
    if mask.any():
        placeholder_hits += int(mask.sum())
        df.loc[mask, c] = np.nan
print(f"Replaced {placeholder_hits:,} '--' placeholder values with NaN across {len(obj_cols_initial)} object columns")


**What just happened:** this scans every text column for the literal string `"--"` and swaps it for a real `NaN`, without touching row or column counts. Watch `back_legroom` and `bed_height` in particular -- their missing-value percentage will visibly jump once this runs, since those blanks were hiding in plain sight under a string that looked like real data.

### 2.2 Boolean flags stored three different ways
The EDA found the same true/false concept stored as a real `bool`, as `object` + NaN,
and as an all-empty `float64` -- plus two columns (`is_cpo`, `is_oemcpo`) where NaN
specifically means False rather than "unknown" (EDA Section 4.3). That distinction
matters: for `fleet`/`frame_damaged`/`has_accidents`/`isCab`/`salvage`/`theft_title`,
the ~46% missing rate is an entire vehicle-history report that wasn't pulled for those
listings -- NaN there means *unknown*, not False. Coercing it to False now would
quietly assume every unreported car is clean, so those six are cast to a nullable
boolean that preserves NaN, and handled explicitly in Section 4's imputation instead.


In [ ]:
# One-sided flags: presence of any value means True, absence means False, by definition.
one_sided_bool_cols = [c for c in ["is_cpo", "is_oemcpo"] if c in df.columns]
for c in one_sided_bool_cols:
    df[c] = df[c].notna()

# Clean flags with negligible missingness -- cast to nullable boolean, filled in Section 4.
clean_bool_cols = [c for c in ["is_new", "franchise_dealer"] if c in df.columns]
for c in clean_bool_cols:
    df[c] = df[c].astype("boolean")

# History-report block -- real missingness (NaN = unknown), kept as NaN for now.
history_block_cols = [c for c in ["fleet", "frame_damaged", "has_accidents",
                                   "isCab", "salvage", "theft_title"] if c in df.columns]
for c in history_block_cols:
    df[c] = df[c].astype("boolean")

print("One-sided (NaN=False):        ", one_sided_bool_cols)
print("Clean booleans:                ", clean_bool_cols)
print("History-block (NaN kept for now):", history_block_cols)


**What just happened:** three different representations collapse into two clean dtypes. `is_cpo`/`is_oemcpo` become real Python `bool` with zero missing values (every NaN just became `False`). The other eight (`is_new`, `franchise_dealer`, and the six history-block columns) become pandas' nullable `boolean` dtype -- same 0/1 values as a normal bool, but still able to hold `<NA>` until Section 4 decides what to do with each group's missingness on purpose, instead of losing that distinction here.

### 2.3 Numeric measurements stored as unit strings
Eight columns hold real numeric measurements with the unit glued onto the string
(e.g. `"34.4 in"`), and `power`/`torque` each pack two numbers into one string (e.g.
`"383 hp @ 5,600 RPM"`) (EDA Section 4.2). A regex pulls the leading number out of
each measurement column; the units are dropped since every row uses the same one, so
they carry no information. **Note:** the power/torque regex assumes the pattern
`<value> <unit> @ <rpm> RPM` -- worth spot-checking a handful of parsed rows against
the raw strings after running, in case the real formatting varies.


In [ ]:
def extract_number(series):
    return pd.to_numeric(series.astype(str).str.extract(r"(-?\d+\.?\d*)")[0], errors="coerce")

unit_cols = ["back_legroom", "front_legroom", "fuel_tank_volume",
             "height", "length", "wheelbase", "width", "maximum_seating"]
unit_cols = [c for c in unit_cols if c in df.columns]
for c in unit_cols:
    df[c] = extract_number(df[c])

# power / torque: "383 hp @ 5,600 RPM" -> numeric value and numeric RPM
for c, val_name, rpm_name in [("power", "power_hp", "power_rpm"),
                               ("torque", "torque_lbft", "torque_rpm")]:
    if c in df.columns:
        parts = df[c].astype(str).str.extract(r"(?P<val>-?\d[\d,]*\.?\d*)\D+(?P<rpm>[\d,]+)")
        df[val_name] = pd.to_numeric(parts["val"].str.replace(",", "", regex=False), errors="coerce")
        df[rpm_name] = pd.to_numeric(parts["rpm"].str.replace(",", "", regex=False), errors="coerce")
        df = df.drop(columns=[c])

print("Parsed unit columns:", unit_cols)
print("Parsed power/torque into:", [c for c in ["power_hp","power_rpm","torque_lbft","torque_rpm"] if c in df.columns])


**What just happened:** eight columns flip from `object` to `float64`, and `power`/`torque` are each replaced by two new numeric columns, for a net column-count change of -2 (two dropped) +4 (four added). Any raw string that didn't match the expected pattern becomes `NaN` here rather than crashing the cell -- those get picked up later by Section 4's group-median imputation, but it's worth spot-checking a few parsed values against the original strings to confirm the regex caught the intended number.

## 3. Outlier handling

Covers the three columns the brief calls out by name -- `mileage`, `price`, `year` --
plus one more numeric error the EDA surfaced along the way (implausible fuel-economy
values). Every row drop below is confirmed against the 50%-combined-drop ceiling at
the end of this section.

**mileage** -- 2 rows carry an odometer sentinel value of 99,999,988, which
single-handedly collapsed the column's standard deviation and masked its true
correlation with price (-0.11 observed vs. -0.45 once removed, per the EDA's Section
4.4). Only 2 rows, unambiguous garbage -- dropped outright rather than imputed.


In [ ]:
mileage_sentinel_mask = df["mileage"] > 500_000
n_mileage_dropped = int(mileage_sentinel_mask.sum())
df = df.loc[~mileage_sentinel_mask]
print(f"Dropped {n_mileage_dropped} mileage-sentinel rows")


**Running total:** 301,380 -> roughly 301,378 after dropping the 2 sentinel rows. Smallest row-level fix in the whole notebook by count, and the most consequential by effect -- recall from the EDA that these same 2 rows were masking mileage's true correlation with price.

**price** -- 379 listings sit under \$1,000 or over \$200,000 (EDA Section 5.2). These
aren't data-entry errors -- a few real cars really do cost \$1.75M -- but they're rare
and extreme enough that a squared-error model trained on the full range would be
dominated by a handful of them. Dropped here rather than capped, since capping would
silently misrepresent a real price as a fake one.


In [ ]:
price_extreme_mask = (df["price"] < 1_000) | (df["price"] > 200_000)
n_price_dropped = int(price_extreme_mask.sum())
df = df.loc[~price_extreme_mask]
print(f"Dropped {n_price_dropped} price-extreme rows")


**Running total:** -> roughly 300,999 rows after dropping the 379 price-extreme listings. That's the last row-level drop in the notebook -- everything from here forward only adds or modifies columns, never removes rows.

**year** -- no impossible values turned up (range is 1923-2021, and the low end is
genuine classic-car listings, not typos). Rather than drop real data, `year` is left
as-is here and handled instead through the `vehicle_age` feature in Section 5, which
naturally compresses the effect of a handful of very old listings without deleting them.


In [ ]:
print("year range:", int(df["year"].min()), "-", int(df["year"].max()),
      "| rows before 1990:", int((df["year"] < 1990).sum()))


**city_fuel_economy** -- 74 gasoline-fuel listings claim over 60 mpg, and 442 rows sit
at exactly 127 mpg, a suspiciously round cap (EDA Section 4.4). Real gasoline cars
don't do this; electric vehicles legitimately do (MPGe is a different scale). Rather
than drop these rows and lose their other columns, the fuel-economy values themselves
are set to missing for the affected gasoline rows and left for Section 4's imputation
to fill with a plausible value.


In [ ]:
bad_mpg_mask = (df["fuel_type"] == "Gasoline") & (df["city_fuel_economy"] > 60)
n_bad_mpg = int(bad_mpg_mask.sum())
df.loc[bad_mpg_mask, ["city_fuel_economy", "highway_fuel_economy"]] = np.nan
print(f"Nulled fuel-economy values on {n_bad_mpg} implausible gasoline rows")


**What just happened:** no rows are removed here -- the two fuel-economy columns just lose their (implausible) values on 74 specific rows, dropping back to `NaN` so Section 4's imputation can replace them with a value consistent with that vehicle's actual fuel type.

**is_new** -- 4,406 listings are flagged `is_new = True` with over 1,000 miles on the
odometer (EDA Section 4.5). Mileage is the more reliable signal of the two, so rather
than drop these rows, `is_new` is corrected to False for them -- preserving the rest of
each row's information instead of discarding it.


In [ ]:
is_new_conflict_mask = df["is_new"].fillna(False) & (df["mileage"] > 1000)
n_isnew_fixed = int(is_new_conflict_mask.sum())
df.loc[is_new_conflict_mask, "is_new"] = False
print(f"Corrected is_new on {n_isnew_fixed} rows where it contradicted mileage")


**What just happened:** again, no rows removed -- `is_new` flips from `True` to `False` on the 4,406 affected rows. Every other column on those rows is untouched, which is the whole point of fixing in place instead of dropping.

**Confirming the 50% ceiling** -- every row drop above (duplicates, mileage sentinel,
price extremes) as a share of the original raw row count. `is_new`/fuel-economy fixes
don't remove rows, so they aren't counted here.


In [ ]:
n_after_row_drops = len(df)
total_dropped = n_raw - n_after_row_drops
pct_dropped = total_dropped / n_raw * 100
print(f"Rows dropped so far: {total_dropped:,} of {n_raw:,} raw rows ({pct_dropped:.2f}%)")
assert pct_dropped < 50, "Combined null/outlier row-dropping exceeds the 50% ceiling"


**What just happened:** by this point the notebook has dropped roughly 1,591 rows total (1,210 + 2 + 379) out of 302,590 -- about 0.53% of the data, nowhere close to the 50% ceiling. That headroom is exactly why the more conservative choices elsewhere (fixing `is_new` in place instead of dropping, nulling two columns instead of the whole row) were affordable without needing it.

## 4. Imputation

Strategy is grouped by *why* each set of columns is missing, per the EDA's Section 2 findings, rather than one blanket rule.

**History-report columns** (`fleet`, `frame_damaged`, `has_accidents`, `isCab`,
`salvage`, `theft_title`) -- ~46% missing because a vehicle-history report wasn't
pulled for those listings, not because the underlying fact is unknowable. Filling
with the mode (almost certainly False, since accidents/salvage/theft are rare) would
quietly assume every unreported car is clean -- an optimistic, unverifiable guess.
Instead, a `history_report_available` indicator is added first, *then* each flag is
filled with False for modeling purposes -- so the model can learn to weight the flag
differently on rows where it's really just a default rather than a verified fact.


In [ ]:
history_block_present = [c for c in history_block_cols if c in df.columns]
df["history_report_available"] = df[history_block_present].notna().all(axis=1)
for c in history_block_present:
    df[c] = df[c].fillna(False)
print("Added history_report_available; filled history-block flags with False where missing")


**What just happened:** one new column (`history_report_available`) captures whether a row had a report *before* the fix, then all six history-block columns lose their `<NA>` values in favor of `False`. Net effect: +1 column, zero rows changed.

**Truck-only fields** (`bed`, `bed_length`, `cabin`) -- missing because the vehicle
isn't a truck, not because data collection failed (EDA Section 2). A `has_bed`
indicator is captured before filling, then the categorical fields get an explicit
"Not Applicable" category and the numeric field gets 0 -- both defensible only because
the missingness itself already tells the model "this isn't a truck."


In [ ]:
if "bed" in df.columns:
    df["has_bed"] = df["bed"].notna()
for c in ["bed", "cabin"]:
    if c in df.columns:
        df[c] = df[c].fillna("Not Applicable")
if "bed_length" in df.columns:
    df["bed_length"] = df["bed_length"].fillna(0)
print("Added has_bed indicator; filled truck-only fields")


**What just happened:** one new `has_bed` column captures the pre-fill truck/non-truck signal, `bed` and `cabin` gain an explicit `"Not Applicable"` category, and `bed_length` gains a `0`. No rows or other columns are touched.

**is_cpo / is_oemcpo** were already resolved in Section 2.2 (NaN = False by
definition). **is_new / franchise_dealer** have negligible missingness -- filled with
the column mode rather than a more elaborate strategy, since there's no structured
pattern here to exploit.


In [ ]:
for c in clean_bool_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0])
print("Mode-filled:", clean_bool_cols)


**Physical measurement columns** (`back_legroom`, `front_legroom`, `height`, `length`,
`wheelbase`, `width`, `fuel_tank_volume`, `maximum_seating`, plus the parsed
`power_hp`/`power_rpm`/`torque_lbft`/`torque_rpm`) vary systematically by vehicle
shape -- a sedan's legroom looks nothing like a pickup's. A single global median
would blur that away, so each is imputed with the median *within its `body_type`*
first, falling back to the global median only if an entire `body_type` group is
missing that column.


In [ ]:
physical_cols = [c for c in unit_cols + ["power_hp", "power_rpm", "torque_lbft", "torque_rpm"]
                  if c in df.columns]
for c in physical_cols:
    group_median = df.groupby("body_type")[c].transform("median")
    df[c] = df[c].fillna(group_median)
    df[c] = df[c].fillna(df[c].median())
print("Group-median (by body_type) imputed:", physical_cols)


**What just happened:** each column keeps its existing values wherever it already had one -- only the genuinely missing cells get replaced, and with a `body_type`-specific median rather than one number for the whole dataset. Concretely: a missing `height` on a listed SUV gets filled with the *typical SUV's* height, not the typical car's height across all body types combined.

**Fuel-economy columns** (`city_fuel_economy`, `highway_fuel_economy`) vary far more by
powertrain than by body shape -- an electric sedan and a gasoline sedan aren't
comparable on this axis -- so these are imputed with the median *within `fuel_type`*
instead, which also naturally repairs the values nulled out in Section 3 for the
implausible gasoline rows.


In [ ]:
mpg_cols = [c for c in ["city_fuel_economy", "highway_fuel_economy"] if c in df.columns]
for c in mpg_cols:
    group_median = df.groupby("fuel_type")[c].transform("median")
    df[c] = df[c].fillna(group_median)
    df[c] = df[c].fillna(df[c].median())
print("Group-median (by fuel_type) imputed:", mpg_cols)


**What just happened:** same mechanism as above, grouped by `fuel_type` instead -- and this is also where the 74 rows nulled out in Section 3 get their fuel-economy values replaced, this time with a number that's actually plausible for a gasoline vehicle instead of an electric-scale reading.

**Remaining numeric columns** (`owner_count`, `seller_rating`, `daysonmarket`, and
anything else left over) -- filled with the column median as a simple, defensible
default; none showed the kind of structured missingness that would justify a more
elaborate strategy the way the columns above did.


In [ ]:
remaining_numeric = df.select_dtypes(include=[np.number]).columns
filled_numeric = []
for c in remaining_numeric:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())
        filled_numeric.append(c)
print("Median-filled remaining numeric gaps in:", filled_numeric)


**Remaining categorical columns** (e.g. sparse color or trim entries) get an explicit
`"Unknown"` category rather than the mode -- silently guessing a specific make or
color would fabricate information that isn't there. `listing_color`'s existing
`"UNKNOWN"` values are left as-is; they already mean exactly this.


In [ ]:
remaining_object = df.select_dtypes(include="object").columns
filled_object = []
for c in remaining_object:
    if df[c].isna().any():
        df[c] = df[c].fillna("Unknown")
        filled_object.append(c)
print("Unknown-filled remaining categorical gaps in:", filled_object)


## 5. Feature engineering

No engineered feature below uses `price` or any column derived from it.

**vehicle_age** (years since the newest model year in the dataset) and
**mileage_per_year** (a usage-intensity signal -- two cars with identical mileage tell
a different story if one is 2 years old and the other is 10). This is also how `year`'s
outlier handling is addressed per Section 3: rather than dropping rare old listings,
`vehicle_age` naturally compresses their effect through scaling in Section 8.


In [ ]:
REFERENCE_YEAR = int(df["year"].max())  # newest model year present in the data
df["vehicle_age"] = REFERENCE_YEAR - df["year"]
df["mileage_per_year"] = df["mileage"] / df["vehicle_age"].replace(0, 1)  # avoid divide-by-zero for brand-new cars
print("Added vehicle_age, mileage_per_year")


**What just happened:** two new numeric columns. Since the newest model year in the data is 2021, `vehicle_age` ranges from 0 (a brand-new 2021 listing) up to 98 (the 1923 classic-car listing found in the EDA) -- a more model-friendly quantity than raw `year`, since 'how old is this car' matters more directly to price than 'what calendar year is it.'

**listed_date** -- stored as a string; parsed to a real date and broken into
`listing_month` (seasonality) and `listing_dow` (day of week a listing went live),
since the raw calendar date itself would behave like an identifier and wouldn't
generalize to future listings the way a month or weekday can.


In [ ]:
if "listed_date" in df.columns:
    parsed_date = pd.to_datetime(df["listed_date"], errors="coerce")
    df["listing_month"] = parsed_date.dt.month
    df["listing_dow"] = parsed_date.dt.dayofweek
    df = df.drop(columns=["listed_date"])
    for c in ["listing_month", "listing_dow"]:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].mode().iloc[0])
    print("Parsed listed_date into listing_month, listing_dow")


**What just happened:** `listed_date` disappears entirely, replaced by two small integer columns (`listing_month` 1-12, `listing_dow` 0-6). Net column count: -1 date column, +2 new ones.

**major_options** -- stored as a delimited string of feature names per listing (e.g.
`"Bluetooth, Backup Camera, Navigation System"`). Parsed into `n_options` (a count --
more-equipped cars likely command a premium) plus individual binary flags for the 10
most common options, so the model can weight specific in-demand features rather than
only a total count. **Note:** the parsing below assumes a comma-separated,
bracket-wrapped string; spot-check `df["major_options"]`'s raw format on a few rows
before running if that assumption looks off.


In [ ]:
if "major_options" in df.columns:
    options_series = (df["major_options"].fillna("")
                       .str.strip("[]")
                       .str.replace("'", "", regex=False)
                       .str.split(","))
    options_series = options_series.apply(lambda lst: [o.strip() for o in lst if o.strip()])

    df["n_options"] = options_series.apply(len)

    all_options = pd.Series([o for lst in options_series for o in lst])
    top_options = all_options.value_counts().head(10).index.tolist()
    for opt in top_options:
        col_name = "opt_" + re.sub(r"[^0-9a-zA-Z]+", "_", opt).strip("_").lower()
        df[col_name] = options_series.apply(lambda lst, opt=opt: opt in lst)

    df = df.drop(columns=["major_options"])
    print("Top options encoded as flags:", top_options)


**What just happened:** `major_options` disappears, replaced by `n_options` plus up to 10 new `opt_*` binary columns -- one per option common enough to make the top-10 cut. Exactly which options end up there depends on your actual data, so the printed list above is worth a glance before moving on.

## 6. Encoding

### 6.1 Label-encoding binary columns
Every boolean column built or cleaned in Sections 2-5 is a true binary -- mapped
straight to 0/1.


In [ ]:
bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns.tolist()
for c in bool_cols:
    df[c] = df[c].astype(int)
print(f"Label-encoded {len(bool_cols)} binary columns: {bool_cols}")


**What just happened:** every column identified as boolean up to this point (the original flags, plus `history_report_available` and `has_bed` from Sections 4-5) converts from `True`/`False` to `1`/`0`. Values don't change, only their dtype and display.

### 6.2 One-hot encoding categorical columns
Every remaining `object`-dtype column gets the brief's exact rule: the single most
frequent category becomes the implicit reference (no dummy column), the next 20 most
frequent categories each get a dummy column, and anything rarer than that is dropped
rather than encoded -- the alternative (thousands of one-listing dummy columns for
`model_name` or `trim_name`) would blow up the feature space for no real gain. The
raw column itself is left untouched here (not dropped) so it survives into Section 7
for the fairness audit.


In [ ]:
def top20_dummies(frame, col, max_dummies=20):
    counts = frame[col].value_counts()
    reference = counts.index[0]                    # most frequent -> implicit baseline
    keep = counts.index[1:1 + max_dummies]           # next 20 most frequent -> dummy columns
    dummies = pd.get_dummies(frame[col].where(frame[col].isin(keep)), prefix=col)
    return dummies, reference, keep.tolist()

categorical_cols = df.select_dtypes(include="object").columns.tolist()

encoded_blocks = []
encoding_log = []
for col in categorical_cols:
    dummies, reference, kept = top20_dummies(df, col)
    encoded_blocks.append(dummies)
    encoding_log.append({"column": col, "reference_category": reference, "n_dummies": dummies.shape[1]})

df = pd.concat([df] + encoded_blocks, axis=1)
pd.DataFrame(encoding_log)


**What just happened:** each categorical column keeps its original values untouched (needed for Section 7) and gains up to 20 new `col_CategoryName` dummy columns alongside it. The table this cell produces shows exactly which category became the silent reference for each column -- the one *without* a dummy -- worth knowing before anyone interprets a model coefficient later.

## 7. Preserving subgroup metadata

The raw, un-encoded version of every column that just got one-hot encoded
(`categorical_cols`) is still sitting in `df` untouched -- Section 6.2 added dummy
columns alongside them rather than replacing them. That's exactly what the brief
requires: these columns must be present in the exported dataset for Notebook 5's
fairness audit, but must never enter `X`/`y` in Notebooks 3-4. `metadata_cols` is
saved here as the exclusion list every later notebook should reuse.


In [ ]:
metadata_cols = categorical_cols
print(f"{len(metadata_cols)} metadata columns retained (excluded from modeling X/y):")
print(metadata_cols)


## 8. Scaling

Every non-metadata feature is standard-scaled except `price`, which stays in its
original units per the brief -- it's the actual regression target in dollars, not a
z-score. This includes the one-hot dummy and label-encoded binary columns, since the
brief specifies "all non-metadata features," not just the continuous ones.


In [ ]:
feature_cols = [c for c in df.columns if c not in metadata_cols and c != "price"]

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
print(f"Standard-scaled {len(feature_cols)} feature columns")
print(f"Left untouched: price, and {len(metadata_cols)} metadata columns")


**What just happened:** every feature column now has mean 0 and standard deviation 1 -- including the 0/1 dummy and label-encoded columns, per the brief's literal instruction. A `body_type_SUV` value of, say, `1.8` doesn't mean 'very SUV'; it means 'this row is an SUV, and SUVs happen to be less common than the reference category' -- just an artifact of scaling a binary column, worth knowing if it looks confusing later.

## 9. Split and export

80/20 split with `random_state=42` per the brief -- Notebook 3 reuses this exact
split. Saved to the local class data folder (next to the raw CSV) rather than the
repo, since the full dataset is far over GitHub's 100 MB limit.


In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_path = OUTPUT_DIR / "processed_train.csv"
test_path = OUTPUT_DIR / "processed_test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train: {train_df.shape} -> {train_path}")
print(f"Test:  {test_df.shape} -> {test_path}")


**What just happened:** the roughly 300,999 rows remaining after Section 3 split into about 240,799 for training and 60,200 for testing -- an 80/20 split, fixed by `random_state=42` so Notebook 3 reproduces these exact same rows on its side.

## Sanity checks before handoff to Notebook 3
A quick validation pass on the exported split -- catches a broken assumption now
rather than three notebooks from now when a metric looks suspiciously perfect (or
suspiciously bad).


In [ ]:
# No leftover missing values anywhere in either export
train_nan_cols = train_df.columns[train_df.isna().any()].tolist()
test_nan_cols = test_df.columns[test_df.isna().any()].tolist()
assert not train_nan_cols, f"NaNs remain in train: {train_nan_cols}"
assert not test_nan_cols, f"NaNs remain in test: {test_nan_cols}"
print("No missing values in either processed_train or processed_test")


Confirms three things the rest of the pipeline depends on: `price` is still in real
dollars (not accidentally scaled), the metadata columns needed for Notebook 5's
fairness audit are present in both files, and none of the label-leakage columns
identified in Section 1 slipped back in.


In [ ]:
# price left in original units, not standard-scaled
assert train_df["price"].min() > 100, "price looks scaled, not left in original units"
print("price range in train:", train_df["price"].min(), "-", train_df["price"].max())

# metadata columns present in both exports
missing_meta_train = [c for c in metadata_cols if c not in train_df.columns]
missing_meta_test = [c for c in metadata_cols if c not in test_df.columns]
assert not missing_meta_train and not missing_meta_test, "metadata columns missing from an export"
print(f"All {len(metadata_cols)} metadata columns present in both train and test")

# dropped/leakage columns didn't sneak back in
leakage_check = [c for c in ["savings_amount", "listing_id", "sp_id", "Unnamed: 0"] if c in train_df.columns]
assert not leakage_check, f"leakage/ID columns present in export: {leakage_check}"
print("No leakage or ID columns present in the export")


Last check: the EDA found `horsepower` as the strongest linear driver of price
(r = 0.61). Standard-scaling is a linear transform, so that correlation should be
essentially unchanged after preprocessing -- if it isn't, something in Sections 2-8
altered the relationship between the two rather than just its units.


In [ ]:
if "horsepower" in train_df.columns:
    r = train_df["horsepower"].corr(train_df["price"])
    print(f"horsepower vs price correlation after preprocessing: {r:.3f}  (EDA raw value: 0.61)")


Final shape and split-ratio check.


In [ ]:
print(f"train: {train_df.shape}, test: {test_df.shape}")
print(f"train fraction: {len(train_df) / (len(train_df) + len(test_df)):.3f}  (target: 0.80)")
print(f"feature columns: {len(feature_cols)}  |  metadata columns: {len(metadata_cols)}  |  target: price")


## Summary for Notebook 3
- **Target:** `price` (untouched, original dollar units)
- **Metadata columns to exclude from X/y:** the object-dtype columns listed in
  `metadata_cols` above -- present in both exported CSVs for Notebook 5's fairness
  audit, but must be dropped before constructing `X`
- **Everything else** is numeric and already scaled, ready to model directly in
  Notebook 3
